# Automated License Plate Character Recognition under Rain, Fog, and Glare
### **Group 6**: Aditya Verma (12408804), Varun Singh Rajpurohit (12409154), Appan (12409471)

**Objective:** Develop a robust automatic license plate recognition (ALPR) system resilient to severe adverse weather conditions (fog, rain streaks, and headlight glare) using a **Dark Channel Prior (DCP) de-hazing pre-filter**, **specular glare suppression**, and a **lightweight PyTorch CNN character recognizer**.

## 1. Environment Setup & Dependencies
Runs seamlessly on free Google Colab (CPU or T4 GPU) or local laptop CPU.

In [ ]:
!pip install -q torch torchvision opencv-python matplotlib numpy scipy tqdm

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.append("..")
sys.path.append(".")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Active PyTorch Compute Device: {device}")

## 2. Generate Adverse Weather Benchmark Samples
Generates clean vehicle plates along with synthetic Fog (Koschmieder atmospheric scattering), Rain streaks, Headlight Glare, and Combined weather degradations.

In [ ]:
from generate_sample_data import generate_benchmark_dataset

generate_benchmark_dataset(output_dir="../data/samples", num_samples=6)

## 3. Train Lightweight CNN Character Recognizer
Trains a <300k parameter CNN for 36-class alphanumeric recognition (0-9, A-Z) with fast convergence.

In [ ]:
from train import train_model

weights_path = "../models/char_cnn_lightweight.pth"
train_model(
    samples_per_class=350,
    epochs=8,
    batch_size=64,
    save_path=weights_path,
    device=device
)

## 4. Run End-to-End Adverse Weather ALPR Pipeline
Test on severe adverse weather (Combined Fog + Rain + Glare) and visualize step-by-step restoration and OCR.

In [ ]:
from src.pipeline import AdverseWeatherALPRPipeline
from src.utils.visualization import plot_pipeline_comparison

pipeline = AdverseWeatherALPRPipeline(weights_path=weights_path, device=device)

test_img_path = "../data/samples/sample_01_combined.jpg"
test_img = cv2.imread(test_img_path)

res = pipeline.process(test_img, with_enhancement=True)

print(f"Ground Truth:      KA05MC2024")
print(f"Predicted Plate:   {res['plate_text']}")
print(f"Detected BBox:     {res['plate_bbox']}")

plot_pipeline_comparison(
    raw_img=test_img,
    enhanced_img=res["work_image"],
    dark_channel=res["enhancement_data"]["dark_channel"],
    transmission_map=res["enhancement_data"]["transmission_map"],
    plate_bbox=res["plate_bbox"],
    plate_crop=res["plate_crop"],
    char_predictions=res["char_preds"],
    ground_truth="KA05MC2024"
)

## 5. Quantitative Ablation Study & Evaluation
Compares ALPR accuracy and perceptual quality (PSNR, SSIM, Character Error Rate) with vs. without the pre-filter.

In [ ]:
from evaluate import evaluate_benchmark

evaluate_benchmark(data_dir="../data/samples", weights_path=weights_path, device=device)